## imports:

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)


# part C :

In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
PROC, ART, REP = ROOT/"data/processed", ROOT/"artifacts/tfidf", ROOT/"reports"
ART.mkdir(parents=True, exist_ok=True); REP.mkdir(exist_ok=True)

SPLIT_SEED = 2026
REC_CLASSES  = ["recommended", "not_recommended", "no_idea"]
BAND_CLASSES = ["low", "medium", "high"]
BAND_CODE    = {"low": 0, "medium": 1, "high": 2}

def load(name):   
    return pd.read_csv(PROC/name, dtype=str, keep_default_na=False, na_values=[""])

train, val, test = load("train_supervised.csv"), load("validation.csv"), load("test.csv")
for n, d in [("train", train), ("val", val), ("test", test)]:
    print(n, len(d), "| rec labeled:", d["recommendation_status"].notna().sum(),
          "| band labeled:", d["rating_band"].notna().sum())

train 3555676 | rec labeled: 2944940 | band labeled: 3271658
val 917793 | rec labeled: 788823 | band labeled: 843574
test 917785 | rec labeled: 788823 | band labeled: 843574


### TF-IDF:

In [3]:
VEC_PATH = ART / "tfidf_vectorizer.joblib"
if VEC_PATH.exists() and (ART/"X_test.npz").exists():
    vec = joblib.load(VEC_PATH)
    X_train, X_val, X_test = (sp.load_npz(ART/f"X_{s}.npz") for s in ("train", "val", "test"))
else:
    t0 = time.time()
    vec = TfidfVectorizer(
        ngram_range=(1, 2), min_df=5, max_features=400_000, sublinear_tf=True,
        dtype=np.float32, token_pattern=r"(?u)[\w\u200c]{2,}")   
    vec.fit(train["body"].sample(500_000, random_state=SPLIT_SEED))   #
    print("vocab size:", len(vec.vocabulary_), f"({time.time()-t0:.0f}s)")

    def transform(texts, chunk=200_000):
        parts = Parallel(n_jobs=min(4, os.cpu_count()))(
            delayed(vec.transform)(texts[i:i+chunk]) for i in range(0, len(texts), chunk))
        return sp.vstack(parts, format="csr")

    X_train = transform(train["body"].tolist())
    X_val, X_test = transform(val["body"].tolist()), transform(test["body"].tolist())
    joblib.dump(vec, VEC_PATH)
    for s, X in (("train", X_train), ("val", X_val), ("test", X_test)):
        sp.save_npz(ART/f"X_{s}.npz", X, compressed=False)
    print(f"done in {time.time()-t0:.0f}s")

print(X_train.shape, X_train.nnz)
train = train.drop(columns=["body"], errors="ignore")   

vocab size: 115840 (45s)
done in 165s
(3555676, 115840) 53136203


### metrics:

In [ ]:
def evaluate(y_true, y_pred, classes, ordinal=False):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=classes, zero_division=0)
    out = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f.mean()), "macro_precision": float(p.mean()), "macro_recall": float(r.mean()),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "per_class": {c: {"precision": float(p[i]), "recall": float(r[i]), "f1": float(f[i]),
                          "support": int(s[i])} for i, c in enumerate(classes)},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=classes).tolist(),
    }
    if ordinal:
        t = pd.Series(y_true).map(BAND_CODE).values; q = pd.Series(y_pred).map(BAND_CODE).values
        out["ordinal_distance_error"] = float(np.abs(t - q).mean())
    return out

TASKS = {"recommendation": ("recommendation_status", REC_CLASSES, False),
         "rating_band":    ("rating_band", BAND_CLASSES, True)}
def make_clf(alpha):
    return SGDClassifier(loss="log_loss", alpha=alpha, max_iter=15, tol=1e-3,
                         class_weight="balanced", random_state=SPLIT_SEED)

In [ ]:
ALPHAS, TUNE_N = [3e-7, 1e-6, 3e-6, 1e-5, 3e-5], 300_000
best_alpha_full, tuning = {}, {}
for task, (col, classes, _) in TASKS.items():
    idx_tr = np.flatnonzero(train[col].notna().values)
    rng = np.random.RandomState(SPLIT_SEED)
    sub = np.sort(rng.choice(idx_tr, min(TUNE_N, len(idx_tr)), replace=False))
    Xs, ys = X_train[sub], train[col].values[sub]
    idx_va = np.flatnonzero(val[col].notna().values)
    Xv, yv = X_val[idx_va], val[col].values[idx_va]

    rows = []
    for a in ALPHAS:
        clf = make_clf(a).fit(Xs, ys)
        f1 = f1_score(yv, clf.predict(Xv), labels=classes, average="macro", zero_division=0)
        rows.append({"alpha": a, "val_macro_f1": round(f1, 4)}); print(task, a, round(f1, 4))
    tuning[task] = pd.DataFrame(rows)
    best = tuning[task].sort_values("val_macro_f1").iloc[-1]["alpha"]
    # alpha با تعداد نمونه رابطه‌ی معکوس داره: داده‌ی بیشتر → alpha کوچک‌تر
    best_alpha_full[task] = float(best) * len(sub) / len(idx_tr)
    print(f"→ {task}: best alpha on sample = {best}, scaled for full train = {best_alpha_full[task]:.2e}\n")

In [ ]:
models, results = {}, {}
for task, (col, classes, ordinal) in TASKS.items():
    idx_tr = np.flatnonzero(train[col].notna().values)
    clf = make_clf(best_alpha_full[task]).fit(X_train[idx_tr], train[col].values[idx_tr])
    models[task] = clf
    results[task] = {}
    for split, X, df in (("validation", X_val, val), ("test", X_test, test)):
        idx = np.flatnonzero(df[col].notna().values)
        y_true, pred = df[col].values[idx], clf.predict(X[idx])
        proba = clf.predict_proba(X[idx])
        results[task][split] = evaluate(y_true, pred, classes, ordinal)
        pr = pd.DataFrame({"id": df["id"].values[idx], "y_true": y_true, "y_pred": pred})
        for i, c in enumerate(clf.classes_):
            pr[f"p_{c}"] = proba[:, i]
        pr.to_parquet(REP/f"preds_tfidf_{task}_{split}.parquet", index=False)
    r = results[task]["test"]
    print(f"{task} TEST | macro_f1={r['macro_f1']:.4f} acc={r['accuracy']:.4f} "
          f"bal_acc={r['balanced_accuracy']:.4f}" +
          (f" ord_err={r['ordinal_distance_error']:.4f}" if ordinal else ""))
    for c, m in r["per_class"].items():
        print(f"   {c:16s} P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f} n={m['support']}")

for task, m in models.items():
    joblib.dump(m, ART/f"logreg_{task}.joblib")
json.dump({"best_alpha_full": best_alpha_full, "tuning_grid": ALPHAS, "tune_n": TUNE_N,
           "vectorizer": {"ngram": [1, 2], "min_df": 5, "max_features": 400000,
                          "fit_sample": 500000, "sample_seed": SPLIT_SEED},
           "results": results},
          open(REP/"baseline_metrics.json", "w"), indent=2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (task, (col, classes, _)) in zip(axes, TASKS.items()):
    cm = np.array(results[task]["test"]["confusion_matrix"], dtype=float)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(cmn, vmin=0, vmax=1)
    ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=30)
    ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", color="white" if cmn[i, j] < .5 else "black")
    ax.set_title(f"{task} (row-normalized)"); ax.set_xlabel("pred"); ax.set_ylabel("true")
plt.tight_layout(); plt.savefig(REP/"fig_baseline_confusion.png", dpi=150); plt.show()

In [ ]:
terms = np.array(vec.get_feature_names_out())
for task, clf in models.items():
    print("==", task)
    for i, c in enumerate(clf.classes_):
        top = np.argsort(-clf.coef_[i])[:15]
        print(f"  {c}: {list(terms[top])}")